In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Aircraft maintenance & departures

Each aircraft has its own maintenance history. For each departure, attach the most recent maintenance check **for that aircraft**. Figure out the right merge and parameters yourself.

1. Merge.
2. How many departures followed a type `'A'` check? A type `'B'`? A type `'C'`?
3. Named-agg groupby by `check_type`: mean `passengers` and mean `duration_h`.
4. `np.corrcoef` on `duration_h` and `passengers` — do longer maintenance checks tend to precede higher-capacity flights?

In [10]:
maintenance = pd.DataFrame({
    'date':        pd.to_datetime(['2023-01-05','2023-02-12','2023-03-20',
                                   '2023-01-08','2023-02-28','2023-04-01']),
    'aircraft_id': ['AA1','AA1','AA1','BB2','BB2','BB2'],
    'check_type':  ['A', 'B', 'A', 'A', 'C', 'B'],
    'duration_h':  [2, 8, 2, 2, 16, 8],
})

departures = pd.DataFrame({
    'date':        pd.to_datetime(['2023-01-15','2023-02-20','2023-03-25',
                                   '2023-01-20','2023-03-10','2023-04-05']),
    'aircraft_id': ['AA1','AA1','AA1','BB2','BB2','BB2'],
    'destination': ['LAX','JFK','ORD','MIA','SEA','DEN'],
    'passengers':  [180, 220, 195, 160, 210, 175],
})

# Your code here
maintenance = maintenance.sort_values('date')
departures = departures.sort_values('date')

m = pd.merge_asof(
    departures, 
    maintenance, 
    by = 'aircraft_id',
    on = 'date'
)
print(m['check_type'].value_counts())

g = m.groupby('check_type').agg(
    mp = ('passengers','mean'),
    md = ('duration_h','mean')
)
print(g)

c = np.corrcoef(m['passengers'],m['duration_h'])[0,1]
print(f'correlation is {c:.2f}')
print('positively correlated')

check_type
A    3
B    2
C    1
Name: count, dtype: int64
                    mp    md
check_type                  
A           178.333333   2.0
B           197.500000   8.0
C           210.000000  16.0
correlation is 0.59
positively correlated


---

## Level 2 — Supplier quotes & purchase orders

For each purchase order, find the **closest supplier quote within 14 days** — either before or after the order date. Orders with no quote in that window get NaN. Remember the sorting rule.

1. Merge.
2. How many orders had no quote? Print the buyer and date for each.
3. Add `total_cost = qty * price`. Use `np.nanmean` across all orders.
4. Among matched orders: use `np.argmin` to find which order got the lowest unit price. Use `np.argsort` to rank matched orders by total cost, highest to lowest.

In [23]:
quotes = pd.DataFrame({
    'date':  pd.to_datetime(['2023-03-01','2023-03-15','2023-04-01','2023-04-20','2023-05-10']),
    'price': [142.50, 138.00, 145.00, 136.50, 148.00],
})

orders = pd.DataFrame({
    'date':  pd.to_datetime(['2023-03-05','2023-03-25','2023-04-10','2023-04-30','2023-06-01']),
    'qty':   [100, 150, 80, 200, 120],
    'buyer': ['X', 'Y', 'X', 'Y', 'X'],
})

# Your code here

quotes = quotes.sort_values('date')
orders = orders.sort_values('date')

m = pd.merge_asof(
    orders, 
    quotes, 
    on = 'date',
    tolerance= pd.Timedelta('14D'),
    direction='nearest'
)

print(m['price'].isna().sum(),'is NA')
print(m.loc[m['price'].isna(),'buyer'],m.loc[m['price'].isna(),'date'])
m['total_cost'] = m['price']*m['qty']
print('mean is: ',np.nanmean(m['total_cost']))
print(m.iloc[np.argmin(m['price'])])

mn = m.dropna()
print(mn.iloc[np.argsort(-mn['total_cost'])])

1 is NA
4    X
Name: buyer, dtype: object 4   2023-06-01
Name: date, dtype: datetime64[ns]
mean is:  18725.0
date          2023-04-30 00:00:00
qty                           200
buyer                           Y
price                       136.5
total_cost                27300.0
Name: 3, dtype: object
        date  qty buyer  price  total_cost
3 2023-04-30  200     Y  136.5     27300.0
1 2023-03-25  150     Y  145.0     21750.0
0 2023-03-05  100     X  142.5     14250.0
2 2023-04-10   80     X  145.0     11600.0


---

## Level 3 — Gold vs oil prices

`gold` and `oil` report monthly prices on alternating months. Free-form analysis — fewer guardrails.

1. Build the full monthly timeline.
2. Add `ratio = gold_price / oil_price` (ounces of oil per ounce of gold). Which month had the highest ratio? Lowest?
3. `np.corrcoef` — do gold and oil move together?
4. `np.percentile` on `ratio` — find Q1 and Q3. How many months fall in the top quartile (above Q3)?
5. Find months where gold was above its mean AND oil was below its mean — a favourable setup for a gold-long/oil-short position. Use `np.mean` and a list comprehension.

In [40]:
gold = pd.DataFrame({
    'month':      pd.to_datetime(['2023-01','2023-03','2023-05','2023-07','2023-09','2023-11'], format='%Y-%m'),
    'gold_price': [1924, 1988, 1977, 1965, 1908, 1994],
})

oil = pd.DataFrame({
    'month':     pd.to_datetime(['2023-02','2023-04','2023-06','2023-08','2023-10','2023-12'], format='%Y-%m'),
    'oil_price': [77.4, 79.8, 71.2, 84.6, 82.1, 72.0],
})

# Your code here

m = pd.merge_ordered(gold, oil, on = 'month', fill_method='ffill').dropna()
m['ratio'] = m['gold_price']/m['oil_price']
print(m.loc[m['ratio'].idxmax(),'month'],'has the highest ratio')
print(m.loc[m['ratio'].idxmin(),'month'],'has the lowest ratio')
c = np.corrcoef(m['gold_price'], m['oil_price'])[0,1]
print('correlation is: ',c)
print('slightly negatively correlated')
qs = np.percentile(m['ratio'],q = [25,75])
print((m['ratio']>qs[1]).sum(),'are above Q3')
gm = np.mean(m['gold_price'])
om = np.mean(m['oil_price'])
print([m for m, g,o in zip(m['month'], m['gold_price'], m['oil_price']) if g>gm and o<om])

2023-06-01 00:00:00 has the highest ratio
2023-09-01 00:00:00 has the lowest ratio
correlation is:  -0.39319636555376986
slightly negatively correlated
3 are above Q3
[Timestamp('2023-03-01 00:00:00'), Timestamp('2023-06-01 00:00:00'), Timestamp('2023-07-01 00:00:00'), Timestamp('2023-12-01 00:00:00')]


,month,gold_price,oil_price,ratio
0,2023-01-01,1924,NaN,NaN
1,2023-02-01,1924,77.4,24.857881
2,2023-03-01,1988,77.4,25.684755
3,2023-04-01,1988,79.8,24.912281
4,2023-05-01,1977,79.8,24.774436
5,2023-06-01,1977,71.2,27.766854
6,2023-07-01,1965,71.2,27.598315
7,2023-08-01,1965,84.6,23.226950
8,2023-09-01,1908,84.6,22.553191
9,2023-10-01,1908,82.1,23.239951
